<a href="https://colab.research.google.com/github/npranav545/aiac/blob/main/lab16_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

import sqlite3
import pandas as pd

# Step 1️: Create in-memory SQLite database
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# Step 2️: Schema Generation
cursor.executescript("""
CREATE TABLE Members (
    Member_ID INTEGER PRIMARY KEY,
    Member_Name TEXT NOT NULL,
    Email TEXT UNIQUE,
    Phone TEXT,
    Join_Date DATE
);

CREATE TABLE Books (
    Book_ID INTEGER PRIMARY KEY,
    Title TEXT NOT NULL,
    Author TEXT,
    Genre TEXT,
    Is_Available INTEGER DEFAULT 1   -- 1=True, 0=False
);

CREATE TABLE Loans (
    Loan_ID INTEGER PRIMARY KEY,
    Member_ID INTEGER,
    Book_ID INTEGER,
    Loan_Date DATE,
    Return_Date DATE,
    FOREIGN KEY (Member_ID) REFERENCES Members(Member_ID),
    FOREIGN KEY (Book_ID) REFERENCES Books(Book_ID)
);
""")

# Step 3: Insert Sample Data
cursor.executescript("""
INSERT INTO Members (Member_ID, Member_Name, Email, Phone, Join_Date) VALUES
(1, 'Alice Johnson', 'alice@gmail.com', '9876543210', '2025-06-01'),
(2, 'Bob Smith', 'bob@gmail.com', '9123456780', '2025-06-10'),
(3, 'Charlie Brown', 'charlie@gmail.com', '9012345678', '2025-07-01');

INSERT INTO Books (Book_ID, Title, Author, Genre, Is_Available) VALUES
(101, 'Data Science 101', 'Andrew Ng', 'Technology', 1),
(102, 'The Great Gatsby', 'F. Scott Fitzgerald', 'Fiction', 1),
(103, 'Clean Code', 'Robert C. Martin', 'Programming', 1);

INSERT INTO Loans (Loan_ID, Member_ID, Book_ID, Loan_Date, Return_Date) VALUES
(1001, 1, 101, '2025-10-10', '2025-10-17'),
(1002, 2, 102, '2025-10-11', '2025-10-18'),
(1003, 3, 103, '2025-10-12', NULL);
""")

# Step 4: Basic Query – Books borrowed by a specific member
query = """
SELECT M.Member_Name, B.Title, L.Loan_Date, L.Return_Date
FROM Members M
JOIN Loans L ON M.Member_ID = L.Member_ID
JOIN Books B ON L.Book_ID = B.Book_ID
WHERE M.Member_Name = 'Alice Johnson';
"""
df1 = pd.read_sql_query(query, conn)
print(" Books borrowed by Alice Johnson:\n")
print(df1.to_string(index=False), "\n")

# Step 5: Update Query – mark book as unavailable when borrowed
update_query = """
UPDATE Books
SET Is_Available = 0
WHERE Book_ID = 101;
"""
cursor.execute(update_query)
conn.commit()

print("✅ Book availability updated (Book_ID 101 → Is_Available = 0)\n")

# Step 6️: Verify Update
df2 = pd.read_sql_query("SELECT * FROM Books;", conn)
print(" Current Books Table:\n")
print(df2.to_string(index=False), "\n")

# Step 7️: Delete Query – safely delete a member without active loans
delete_query = """
DELETE FROM Members
WHERE Member_ID = 3
AND Member_ID NOT IN (SELECT Member_ID FROM Loans WHERE Return_Date IS NULL);
"""
cursor.execute(delete_query)
conn.commit()

# Step 8️: Verify Deletion
df3 = pd.read_sql_query("SELECT * FROM Members;", conn)
print(" Members Table after safe delete:\n")
print(df3.to_string(index=False), "\n")

# Step 9️: Close Connection
conn.close()
print(" Lab 16.3 Completed Successfully!")


 Books borrowed by Alice Johnson:

  Member_Name            Title  Loan_Date Return_Date
Alice Johnson Data Science 101 2025-10-10  2025-10-17 

✅ Book availability updated (Book_ID 101 → Is_Available = 0)

 Current Books Table:

 Book_ID            Title              Author       Genre  Is_Available
     101 Data Science 101           Andrew Ng  Technology             0
     102 The Great Gatsby F. Scott Fitzgerald     Fiction             1
     103       Clean Code    Robert C. Martin Programming             1 

 Members Table after safe delete:

 Member_ID   Member_Name             Email      Phone  Join_Date
         1 Alice Johnson   alice@gmail.com 9876543210 2025-06-01
         2     Bob Smith     bob@gmail.com 9123456780 2025-06-10
         3 Charlie Brown charlie@gmail.com 9012345678 2025-07-01 

 Lab 16.3 Completed Successfully!
